In [116]:
import pandas as pd

In [149]:
file_path ='/Users/leonardhaas/code/streamlit/data/raw_data/Zensus2022_Erwerbstaetige.xlsx'

df = pd.read_excel(file_path, sheet_name="Daten", skiprows=2)

In [118]:
df

,Beruf (Berufsgattungen ISCO-08),Unnamed: 1,Stellung im Beruf,Insgesamt,Name Bundesland zum Zensusstichtag (15.05.2022),Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19
0,ICSO-Code,Bezeichnung,NaN,NaN,Baden-Württemberg,Bayern,Berlin,Brandenburg,Bremen,Hamburg,Hessen,Mecklenburg-Vorpommern,Niedersachsen,Nordrhein-Westfalen,Rheinland-Pfalz,Saarland,Sachsen,Sachsen-Anhalt,Schleswig-Holstein,Thüringen
1,Insgesamt,Insgesamt,Insgesamt,41043450,5667810,7024330,1772180,1208030,321090,948980,3048160,723350,3935110,8621290,2025940,470130,1837640,971950,1479240,988230
2,NaN,NaN,"Angestellte, Arbeiter/-innen",35132960,4889260,6001150,1459770,1023910,278260,804010,2598180,624950,3359810,7389160,1717980,404480,1614430,861360,1243640,862620
3,NaN,NaN,Beamtinnen/Beamte,2103550,279480,324640,82790,72350,16120,40610,157820,37290,225610,476590,124960,27600,66290,38220,85660,47520
4,NaN,NaN,Selbstständige mit Beschäftigten,1937420,250570,384680,84820,57430,13300,48790,140940,33520,179970,383040,93280,21190,79560,40990,83730,41620
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2618,NaN,NaN,"Angestellte, Arbeiter/-innen",69470,8080,7890,6460,2750,1130,1880,6850,1110,6120,15870,2890,680,2410,1930,2140,1270
2619,NaN,NaN,Beamtinnen/Beamte,3080,250,680,690,/,/,/,/,/,/,800,/,/,/,/,/,/
2620,NaN,NaN,Selbstständige mit Beschäftigten,710,/,/,/,/,/,/,/,/,/,/,/,/,/,/,/,/
2621,NaN,NaN,Selbstständige ohne Beschäftigte,610,/,/,/,/,/,/,/,/,/,/,/,/,/,/,/,/


In [150]:
# Rename relevant columns
df.rename(columns={
    df.columns[0]: "ISCO-Code",
    df.columns[1]: "Bezeichnung",
    df.columns[2]: "Stellung im Beruf",
    df.columns[3]: "Insgesamt"
}, inplace=True)

# Forward-fill missing values in job classifications
df["ISCO-Code"] = df["ISCO-Code"].ffill()
df["Bezeichnung"] = df["Bezeichnung"].ffill()

In [120]:
total_sum = df['Insgesamt'][1]

In [151]:
total_sum

41043450

In [121]:

# Drop empty values
#df_long = df_long.dropna(subset=["Anzahl"])

df = df.drop(df.index[:7])

In [122]:
df.rename(columns={'Insgesamt':'Anzahl'},inplace=True)

df.rename(columns={
    "Name Bundesland zum Zensusstichtag (15.05.2022)": 'Baden-Württemberg',
    'Unnamed: 5': 'Bayern',
    'Unnamed: 6': 'Berlin',
    'Unnamed: 7': 'Brandenburg',
    'Unnamed: 8': 'Bremen',
    'Unnamed: 9': 'Hamburg',
    'Unnamed: 10': 'Hessen',
    'Unnamed: 11': 'Mecklenburg-Vorpommern',
    'Unnamed: 12': 'Niedersachsen',
    'Unnamed: 13': 'Nordrhein-Westfalen',
    'Unnamed: 14': 'Rheinland-Pfalz',
    'Unnamed: 15': 'Saarland',
    'Unnamed: 16': 'Sachsen',
    'Unnamed: 17': 'Sachsen-Anhalt',
    'Unnamed: 18': 'Schleswig-Holstein',
    'Unnamed: 19': 'Thüringen'
}, inplace=True)


In [123]:
#TODO clean Anzahl

df = df.replace('/', '0')
# Extract rows where 'Anzahl' is not a number using regex
non_numeric_rows = df[~df['Anzahl'].astype(str).str.match(r'^\d+$')]
print(non_numeric_rows)

Empty DataFrame
Columns: [ISCO-Code, Bezeichnung, Stellung im Beruf, Anzahl, Baden-Württemberg, Bayern, Berlin, Brandenburg, Bremen, Hamburg, Hessen, Mecklenburg-Vorpommern, Niedersachsen, Nordrhein-Westfalen, Rheinland-Pfalz, Saarland, Sachsen, Sachsen-Anhalt, Schleswig-Holstein, Thüringen]
Index: []


In [ ]:
total_df = df.query("`Stellung im Beruf` == 'Insgesamt'")

#drop ingesamt rows
df = df[df["Stellung im Beruf"] != "Insgesamt"]

In [154]:
total_df['Anzahl'].astype(int).sum()

np.int64(41042980)

In [125]:
df = df.assign(is_supervisor=0)
#df['is_supervisor'] = df['ISCO-Code'].apply(lambda x: 1 if str(x).startswith('1') else 0)


In [126]:
df = df.assign(self_employed=0)
df.loc[(df['Stellung im Beruf'] == "Selbstständige mit Beschäftigten") | (df['Stellung im Beruf'] == "Selbstständige ohne Beschäftigten"), 'self_employed'] = 1


In [127]:
df = df.assign(n_employees=0)
df.loc[df['Stellung im Beruf'] == "Selbstständige mit Beschäftigten", 'n_employees'] = 1


In [128]:
df.columns

Index(['ISCO-Code', 'Bezeichnung', 'Stellung im Beruf', 'Anzahl',
       'Baden-Württemberg', 'Bayern', 'Berlin', 'Brandenburg', 'Bremen',
       'Hamburg', 'Hessen', 'Mecklenburg-Vorpommern', 'Niedersachsen',
       'Nordrhein-Westfalen', 'Rheinland-Pfalz', 'Saarland', 'Sachsen',
       'Sachsen-Anhalt', 'Schleswig-Holstein', 'Thüringen', 'is_supervisor',
       'self_employed', 'n_employees'],
      dtype='object')

In [129]:
# Define the columns and their new data types
columns_to_convert = {
    'Anzahl': 'int64',
    'Baden-Württemberg': 'int64',
    'Bayern': 'int64',
    'Berlin': 'int64',
    'Brandenburg': 'int64',
    'Bremen': 'int64',
    'Hamburg': 'int64',
    'Hessen': 'int64',
    'Mecklenburg-Vorpommern': 'int64',
    'Niedersachsen': 'int64',
    'Nordrhein-Westfalen': 'int64',
    'Rheinland-Pfalz': 'int64',
    'Saarland': 'int64',
    'Sachsen': 'int64',
    'Sachsen-Anhalt': 'int64',
    'Schleswig-Holstein': 'int64',
    'Thüringen': 'int64'
}

# Convert the columns to the specified data types
df = df.astype(columns_to_convert)

In [130]:
# First, define the self-employed categories
self_employed_categories = [
    'Selbstständige mit Beschäftigten',
    'Selbstständige ohne Beschäftigte'
]

# Define the state columns and Anzahl
all_count_columns = [
    'Anzahl', 'Baden-Württemberg', 'Bayern', 'Berlin', 'Brandenburg', 
    'Bremen', 'Hamburg', 'Hessen', 'Mecklenburg-Vorpommern', 
    'Niedersachsen', 'Nordrhein-Westfalen', 'Rheinland-Pfalz', 
    'Saarland', 'Sachsen', 'Sachsen-Anhalt', 'Schleswig-Holstein', 
    'Thüringen'
]

# Create a filter mask for non-self-employed workers
filter_mask = df['Stellung im Beruf'].isin(self_employed_categories)

# Group by ISCO-Code and sum all count columns for non-self-employed workers
non_self_employed_sum = df[~filter_mask].groupby('ISCO-Code')[all_count_columns].sum().reset_index()

# Add a new category name column
non_self_employed_sum['category'] = 'Arbeiter*innen & Angestellte'

In [155]:
non_self_employed_sum

,ISCO-Code,Anzahl,Baden-Württemberg,Bayern,Berlin,Brandenburg,Bremen,Hamburg,Hessen,Mecklenburg-Vorpommern,Niedersachsen,Nordrhein-Westfalen,Rheinland-Pfalz,Saarland,Sachsen,Sachsen-Anhalt,Schleswig-Holstein,Thüringen,category
0,1111,8800,1630,2130,0,0,0,0,510,0,530,1230,350,0,300,0,0,0,Arbeiter*innen & Angestellte
1,1112,15410,3010,2000,980,550,0,470,1130,300,1960,1820,290,0,800,400,680,280,Arbeiter*innen & Angestellte
2,1113,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Arbeiter*innen & Angestellte
3,1114,6100,460,690,560,230,0,0,390,0,640,2100,0,0,0,0,250,0,Arbeiter*innen & Angestellte
4,1120,244550,32930,43950,12900,7140,1690,7830,20160,4020,20060,52700,9960,1930,10850,4420,7980,4620,Arbeiter*innen & Angestellte
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
431,9624,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Arbeiter*innen & Angestellte
432,9629,72550,8330,8570,7150,2750,1130,1880,6850,1110,6120,16670,2890,680,2410,1930,2140,1270,Arbeiter*innen & Angestellte
433,0110,21970,1150,2170,740,960,0,410,780,780,4620,4450,1680,370,1090,560,1070,620,Arbeiter*innen & Angestellte
434,0210,22020,1640,2210,380,650,280,350,830,1450,4920,3540,1340,330,830,870,1280,700,Arbeiter*innen & Angestellte


In [132]:
self_employed_df = df[filter_mask]
prep_df=pd.concat([non_self_employed_sum,self_employed_df])

In [133]:
prep_df['is_supervisor'] = prep_df['ISCO-Code'].apply(lambda x: 1 if str(x).startswith('1') else 0)

prep_df.loc[prep_df['Stellung im Beruf'] == "Selbstständige mit Beschäftigten", 'n_employees'] = 5

prep_df.loc[(prep_df['Stellung im Beruf'] == "Selbstständige mit Beschäftigten") | (prep_df['Stellung im Beruf'] == "Selbstständige ohne Beschäftigte"), 'self_employed'] = 1

compact_df = prep_df[['ISCO-Code','Stellung im Beruf','is_supervisor','self_employed','n_employees','Anzahl']]

In [134]:
compact_df.loc[:, 'Stellung im Beruf'] = compact_df['Stellung im Beruf'].fillna('Arbeiter*innen & Angestellte')

In [135]:
compact_df.loc[:, 'is_supervisor'] = compact_df['is_supervisor'].fillna(0)
compact_df.loc[:, 'self_employed'] = compact_df['self_employed'].fillna(0)
compact_df.loc[:, 'n_employees'] = compact_df['n_employees'].fillna(0)

In [136]:
compact_df=compact_df.assign(control_work=4)
compact_df=compact_df.assign(control_daily=2)

""" columns_to_convert = ['is_supervisor', 'self_employed']
compact_df[columns_to_convert] = compact_df[columns_to_convert].astype(int) """
compact_df['is_supervisor'] = compact_df['is_supervisor'].astype(int)
compact_df['self_employed'] = compact_df['self_employed'].astype(int)

In [137]:
compact_df.columns

Index(['ISCO-Code', 'Stellung im Beruf', 'is_supervisor', 'self_employed',
       'n_employees', 'Anzahl', 'control_work', 'control_daily'],
      dtype='object')

In [138]:
cleaning_data =compact_df.merge(df[['ISCO-Code','Bezeichnung']],on='ISCO-Code',how='left')

In [139]:
# Define columns that should retain distinct values
distinctive_cols = ['ISCO-Code', 'Stellung im Beruf', 'Bezeichnung', 'Anzahl']

# Drop duplicates based on these columns, keeping the first occurrence
df_cleaned = cleaning_data.drop_duplicates(subset=distinctive_cols, keep='first')


In [148]:
#TODO check for duplicates and semi-duplicates
df_cleaned.to_csv('/Users/leonardhaas/code/streamlit/data/processed_data/digiclass.csv')